In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import numpy as np

#Carico il dataset delle auto
df = pd.read_csv("vehicles_dataset.csv")
X = df.drop(columns=["price", "paint_color"])
#Rimuovo auto con prezzi fuori range (dati non realistici o outlier)
df = df[
    (df["price"] >= 500) &
    (df["price"] <= 100000)
]

#Rimuovo auto con chilometraggio troppo alto (outlier)
df = df[df["odometer"] < 500000]

#Elimino righe senza informazioni fondamentali (price e year)
df = df.dropna(subset=["price", "year"])

#Ulteriore filtro più preciso su prezzo e km
df = df[
    (df["price"].between(500, 80000)) &
    (df["odometer"].between(0, 400000))
]

# Creo una nuova feature: età dell'auto (più utile dell'anno)
df["car_age"] = 2026 - df["year"]

# Trasformo i km in scala logaritmica per ridurre effetto outlier
df["odometer_log"] = np.log1p(df["odometer"])
#target encoding (versione più potente)
df["model_te"] = df.groupby("model")["price"].transform("mean")
df["fuel_te"] = df.groupby("fuel")["price"].transform("mean")
df["state_te"] = df.groupby("state")["price"].transform("mean")
df["type_te"] = df.groupby("type")["price"].transform("mean")
# Calcolo prezzo medio per marca (target encoding)
mean_price = df.groupby("manufacturer")["price"].mean()

# Sostituisco la marca con il suo prezzo medio
df["manufacturer_te"] = df["manufacturer"].map(mean_price)

# Definisco il target (prezzo da prevedere)
# uso log per rendere la distribuzione più stabile
y = np.log1p(df["price"])

# Definisco le feature (tutte tranne price e year)
X = df.drop(columns=["price", "year"])

# Divido i dati in training e test
# training = per imparare
# test = per valutare il modello
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [38]:
X_train = X_train.dropna(subset=['fuel'])
y_train = y_train.loc[X_train.index] 

X_test = X_test.dropna(subset=['fuel'])
y_test = y_test.loc[X_test.index]


#model
X_train = X_train.dropna(subset=['model'])
y_train = y_train.loc[X_train.index] 

X_test = X_test.dropna(subset=['model'])
y_test = y_test.loc[X_test.index]

#transmission
X_train = X_train.dropna(subset=['transmission'])
y_train = y_train.loc[X_train.index] 

X_test = X_test.dropna(subset=['transmission'])
y_test = y_test.loc[X_test.index]

#title_status
X_train = X_train.dropna(subset=['title_status'])
y_train = y_train.loc[X_train.index] 

X_test = X_test.dropna(subset=['title_status'])
y_test = y_test.loc[X_test.index]

#odometer
X_train = X_train.dropna(subset=['odometer'])
y_train = y_train.loc[X_train.index] 

X_test = X_test.dropna(subset=['odometer'])
y_test = y_test.loc[X_test.index]

#manufacturer
X_train = X_train.dropna(subset=['manufacturer'])
y_train = y_train.loc[X_train.index] 

X_test = X_test.dropna(subset=['manufacturer'])
y_test = y_test.loc[X_test.index]

colonne_da_riempire = ['condition', 'cylinders', 'drive', 'size', 'type', 'paint_color']

X_train[colonne_da_riempire] = X_train[colonne_da_riempire].fillna('Unknown')
X_test[colonne_da_riempire] = X_test[colonne_da_riempire].fillna('Unknown')

In [39]:
# tieni solo righe valide per target e feature importanti
df = df.dropna(subset=[
    "price", "fuel", "year", "model",
    "transmission", "title_status",
    "odometer", "manufacturer"
])

# split
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# colonne da riempire (ok il tuo approccio)
colonne_da_riempire = [
    "condition", "cylinders", "drive",
    "size", "type", "paint_color"
]

X_train[colonne_da_riempire] = X_train[colonne_da_riempire].fillna("Unknown")
X_test[colonne_da_riempire] = X_test[colonne_da_riempire].fillna("Unknown")

# colonne categoriche automatiche
categorical_cols = X_train.select_dtypes(include="object").columns

# encoding
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train[categorical_cols] = encoder.fit_transform(
    X_train[categorical_cols]
)

X_test[categorical_cols] = encoder.transform(
    X_test[categorical_cols]
)

# modello

model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=128,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    random_state=42
)
model.fit(X_train, y_train)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model,
    X_train,
    y_train,
    scoring="neg_mean_absolute_error",
    cv=kf
)

print("📊 MAE medio CV:", -scores.mean())

# 🚀 training finale
model.fit(X_train, y_train)

# 🔮 predizione
y_pred = model.predict(X_test)

# 📊 metriche finali
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print("📊 MAE test:", mae)
print("📊 MSE:", mse)
print("📊 RMSE:", rmse)
r2 = r2_score(y_test, y_pred)
print("📊 R²:", r2)

print("Errore medio:", round(error, 2))

C:\Users\rober\AppData\Local\Temp\ipykernel_24568\498156033.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include="object").columns


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006230 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1490
[LightGBM] [Info] Number of data points in the train set: 281259, number of used features: 21
[LightGBM] [Info] Start training from score 19069.933862
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005366 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1489
[LightGBM] [Info] Number of data points in the train set: 225007, number of used features: 21
[LightGBM] [Info] Start training from score 19072.259361
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006428 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is